Keyword and Skill Extraction

In [33]:
import pandas as pd, numpy as np, re, json
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("../data/resumes_cleaned.csv")
print(df.shape)

(166, 3)


In [34]:
def norm(s):
    """Normalize text the same way for resumes, job descriptions and skill names."""
    s = re.sub(r"[^a-z0-9+#.\s]", " ", s.lower())
    return re.sub(r"\s+", " ", s).strip()

SKILLS = {
    "programming": ["python", "java", "c++", "c#", "javascript", "typescript", "php",
                    "ruby", "scala", "sql", "html", "css","js",  "go", "bash", "shell scripting", "matlab"],
    "data_ml": ["machine learning", "deep learning", "nlp", "tensorflow", "keras", "pytorch",
                "scikit-learn", "pandas", "numpy", "tableau", "power bi", "hadoop", "spark",
                "hive", "pig", "etl", "data analysis", "sentiment analysis", "nltk",
                "opencv", "matplotlib", "seaborn"],
    "web_backend": ["django", "flask", "spring", "hibernate", "react", "angular", "node.js",
                    "asp.net", ".net", "jquery", "bootstrap", "rest api"],
    "database": ["mysql", "oracle", "postgresql", "mongodb", "sql server", "sqlserver",
                 "pl/sql", "db2","redis", "cassandra", "elasticsearch"],
    "devops_cloud": ["aws", "azure", "docker", "kubernetes", "jenkins", "git", "linux",
                     "ansible", "terraform"],
    "testing": ["selenium", "junit", "testng", "manual testing", "automation testing",
                "qtp", "loadrunner"],
    "business": ["sap", "project management", "agile", "scrum", "payroll", "recruitment",
                 "sales", "autocad", "blockchain"],
}

SKILLS["legal"] = ["litigation", "legal research", "legal drafting", "criminal law",
                   "civil law", "court", "legal", "arbitration", "contract law"]
SKILLS["hr"] = ["recruitment", "payroll", "onboarding", "employee relations",
                "talent acquisition", "performance management", "hr", "psychology"]
SKILLS["fitness"] = ["fitness training", "personal training", "gym", "nutrition",
                     "sports science", "yoga", "diet planning"]
SKILLS["arts"] = ["painting", "drawing", "sketching", "photoshop", "illustrator",
                  "art direction", "graphic design", "workshop"]
SKILLS["engineering"] = ["autocad", "staad pro", "revit", "solidworks", "matlab", "plc",
                         "scada", "quality control", "quality assurance", "site inspection",
                         "construction", "surveying", "transformer testing", "inspection"]
SKILLS["testing"] += ["test cases", "jira", "regression testing", "appium", "cucumber"]
SKILLS["network"] = ["firewall", "cisco", "ccna", "vpn", "tcp/ip", "siem"]
SKILLS["programming"] += ["js"]
SKILLS["database"] += ["elasticsearch"]
SKILLS["business"] += ["requirements gathering", "supply chain", "logistics", "inventory", "finance"]

SKILLS["legal"] += ["law", "llb", "tax", "taxation", "income tax", "gst", "accounting", "advocate"]
SKILLS["arts"] += ["karate", "martial arts", "art"]
SKILLS["engineering"] += ["electrical testing", "electrical engineering", "testing"]
SKILLS["soft"] = ["problem solving",  "leadership", "teamwork", "analytical"]

SKILLS["ai_nlp"] = ["rag", "llm", "prompt engineering", "vector embeddings",
                    "tf-idf", "transformers", "langchain", "huggingface"]
SKILLS["web_backend"] += ["fastapi"]
SKILLS["devops_cloud"] += ["github", "google colab"]

SKILLS["ai_nlp"] = ["rag", "llm", "prompt engineering", "vector embeddings",
                       "tf-idf", "transformers", "langchain", "huggingface"]
SKILLS["web_backend"] += ["fastapi"]
SKILLS["devops_cloud"] += ["github", "google colab"]

# normalized, longest first so "machine learning" is checked before shorter terms
ALL_SKILLS = sorted({norm(s) for group in SKILLS.values() for s in group},
                    key=len, reverse=True)

def extract_skills(text):
    text = norm(text)
    found = []
    for skill in ALL_SKILLS:
        pattern = r"(?<![a-z0-9+#.])" + re.escape(skill) + r"(?![a-z0-9+#])"
        if re.search(pattern, text):
            found.append(skill)
    return found

df["skills"] = df["clean_resume"].apply(extract_skills)
print(df[["Category", "skills"]].head(10))

       Category                                             skills
0  Data Science  [sentiment analysis, machine learning, elastic...
1  Data Science  [machine learning, python, github, keras, aws,...
2  Data Science  [electrical engineering, machine learning, dat...
3  Data Science  [machine learning, deep learning, analytical, ...
4  Data Science                      [data analysis, python, java]
5  Data Science           [machine learning, matlab, python, html]
6  Data Science       [machine learning, tableau, drawing, python]
7  Data Science  [machine learning, problem solving, deep learn...
8  Data Science  [sentiment analysis, machine learning, deep le...
9  Data Science  [sentiment analysis, machine learning, data an...


In [35]:
# most common skills overall
print(Counter(s for lst in df["skills"] for s in lst).most_common(25))

# resumes where nothing was found
empty = df[df["skills"].str.len() == 0]
print(len(empty), "resumes with no skills found")
print(empty["Category"].value_counts())

[('sql', 55), ('testing', 48), ('java', 43), ('oracle', 33), ('mysql', 30), ('python', 29), ('html', 29), ('sales', 29), ('javascript', 23), ('linux', 23), ('sql server', 22), ('c++', 19), ('css', 18), ('jquery', 17), ('hr', 15), ('machine learning', 14), ('test cases', 13), ('analytical', 12), ('leadership', 12), ('aws', 11), ('sap', 11), ('drawing', 11), ('problem solving', 11), ('selenium', 11), ('hibernate', 11)]
3 resumes with no skills found
Category
Mechanical Engineer    2
DotNet Developer       1
Name: count, dtype: int64


In [36]:
proc = pd.read_csv("../data/resumes_processed.csv")
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
M = tfidf.fit_transform(proc["processed"])
terms = np.array(tfidf.get_feature_names_out())

def top_keywords(i, n=10):
    row = M[i].toarray().ravel()
    return terms[row.argsort()[::-1][:n]].tolist()

for i in [0, 50, 100]:
    print(df["Category"][i], "->", top_keywords(i))

Data Science -> ['fraud', 'js', 'modelling', 'topic', 'question', 'tool technology', 'scikit learn', 'elastic', 'sqlserver', 'sentiment']
Health and fitness -> ['fitness', 'gold gym', 'gold', 'gym', 'trainer', 'sport science', 'fitness trainer', 'personal trainer', 'heart', 'challenging']
Operations Manager -> ['po', 'card', 'pr', 'operation', 'terminal', 'atm', 'adherence', 'email', 'contribution', 'ocean']


In [37]:
from collections import Counter

for cat in ["Advocate", "HR", "Arts", "Health and fitness", "Testing", "Civil Engineer"]:
    idx = empty[empty["Category"] == cat].index
    words = Counter(" ".join(proc.loc[idx, "processed"]).split())
    print(cat, "->", [w for w, c in words.most_common(25)])

Advocate -> []
HR -> []
Arts -> []
Health and fitness -> []
Testing -> []
Civil Engineer -> []


In [38]:
json.dump(ALL_SKILLS, open("../app/skills.json", "w"))
print(len(ALL_SKILLS), "skills saved")

174 skills saved


In [39]:
print("r" in ALL_SKILLS, "litigation" in ALL_SKILLS, "js" in ALL_SKILLS)

False True True


In [40]:
print("communication" in ALL_SKILLS)

False
